# 스몰톡 멀티턴 fine-tuning — 검증된 데이터 묶음 사용

모델의 `block_size=384`와 vocab16000은 유지합니다. 학습 샘플은 최대256토큰이며 QA 중간을 자르지 않습니다. 원본 세션과 합성 대화 템플릿을 기준으로 train/val을 나눠 겹치는 윈도우의 누출을 막았습니다.

짧은 토큰 윈도우 자체가 이해력을 높인다고 보장하지 않습니다. 대화 이력을 유지하고 답변만 학습하며, 검증 손실과 단일턴/멀티턴 비교로 채택합니다.

**최대2에폭**, 검증 손실 미개선1회 시 조기 종료. 모델 파일은 **MyDrive/eric-chatagent/models/** 아래 새 실행 폴더에 저장합니다. 준비 코드의 로컬 점검과 Colab GPU 학습 실행은 별개입니다. 아래 셀을 순서대로 실행하세요.


In [ ]:
!pip -q install sentencepiece huggingface_hub


In [ ]:
TASK = 'smalltalk'

import os, sys, json, math, time, random, hashlib, shutil, zipfile
from pathlib import Path
from datetime import datetime, timezone
from functools import partial
import torch
import sentencepiece as spm
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive/eric-chatagent')
DATA_DIR = DRIVE_ROOT / 'data'
MODELS_DIR = DRIVE_ROOT / 'models'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SEED = 20261024
random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| model outputs:', MODELS_DIR)


## 데이터 준비
`smalltalk_finetune_colab.zip`를 Drive의 `eric-chatagent/data/`에 넣으세요. 없으면 이 ZIP 한 개만 업로드합니다. 이전 CSV/캐시는 사용하지 않습니다.


In [ ]:
BUNDLE_NAME = 'smalltalk_finetune_colab.zip'
EXPECTED_BUNDLE_SHA256 = '2dae0b2c4759795121ff499641554c3e36d891c9a184210c2f428573dfb0d856'
BUNDLE = DATA_DIR / BUNDLE_NAME
if not BUNDLE.is_file() and (DRIVE_ROOT / BUNDLE_NAME).is_file():
    shutil.copyfile(DRIVE_ROOT / BUNDLE_NAME, BUNDLE)
if not BUNDLE.is_file():
    from google.colab import files
    print('Upload exactly:', BUNDLE_NAME)
    uploaded = files.upload()
    if BUNDLE_NAME not in uploaded:
        raise FileNotFoundError('Expected prepared bundle: ' + BUNDLE_NAME)
    shutil.copyfile(Path('/content') / BUNDLE_NAME, BUNDLE)
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256, 'Wrong/stale bundle; use the archive accompanying this notebook'
EXTRACT_DIR = Path('/content') / (TASK + '_finetune_' + EXPECTED_BUNDLE_SHA256[:12])
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for name in archive.namelist():
        if not (EXTRACT_DIR / name).resolve().is_relative_to(EXTRACT_DIR.resolve()):
            raise ValueError('Unsafe ZIP entry')
    archive.extractall(EXTRACT_DIR)
manifest = json.loads((EXTRACT_DIR / 'manifest.json').read_text())
assert manifest['task'] == TASK
SP_MODEL = EXTRACT_DIR / 'sp_korean.model'
assert hashlib.sha256(SP_MODEL.read_bytes()).hexdigest() == manifest['tokenizer_sha256']
sp = spm.SentencePieceProcessor(model_file=str(SP_MODEL))
sys.path.insert(0, str(EXTRACT_DIR / 'runtime'))
from finetune_data import answer_mask, collate_samples, causal_loss, evaluate_token_weighted, save_torch_state
from model_definition import KoreanGPT, block_size
assert block_size == manifest['model_block_size'] == 384
train_records = [json.loads(line) for line in (EXTRACT_DIR / 'train.jsonl').read_text().splitlines()]
val_records = [json.loads(line) for line in (EXTRACT_DIR / 'val.jsonl').read_text().splitlines()]
assert train_records and val_records
assert not {r['source_key'] for r in train_records} & {r['source_key'] for r in val_records}, 'Source-level split leakage'

def encode_records(records):
    samples = []
    for record in records:
        ids = sp.encode(record['text'], out_type=int)
        assert len(ids) == record['token_count'] <= block_size, 'Token budget changed'
        mask = answer_mask(ids, sp)
        assert any(mask[1:]), 'No shifted answer supervision'
        samples.append({'input_ids': ids, 'answer_mask': mask})
    return samples

train_samples = encode_records(train_records)
val_samples = encode_records(val_records)
print('source overlap: 0 | train:', len(train_samples), '| val:', len(val_samples))
for name, samples in [('train', train_samples), ('val', val_samples)]:
    lengths = sorted(len(s['input_ids']) for s in samples)
    print(name, 'tokens:', sum(lengths), 'median:', lengths[len(lengths)//2], 'max:', lengths[-1],
          'answer targets:', sum(sum(s['answer_mask'][1:]) for s in samples))
print(json.dumps(manifest, ensure_ascii=False, indent=2))


## 기존 모델 적재
초기 모델은 별도로 보존하고, 검증한 묶음의 토크나이저를 사용합니다.


In [ ]:
from huggingface_hub import hf_hub_download
PRETRAINED_NAME = 'KoreanGPT_smalltalk.pt'
OUTPUT_NAME = 'KoreanGPT_smalltalk_mt.pt'
# Reuse an explicitly placed original model; otherwise download only that weight file.
CKPT_IN = MODELS_DIR / PRETRAINED_NAME
if not CKPT_IN.is_file():
    CKPT_IN = Path(hf_hub_download('Dongju-00/eric-chatagent',
        filename='src/model/checkpoints/' + PRETRAINED_NAME,
        local_dir=str(MODELS_DIR / 'pretrained')))
model = KoreanGPT(sp.get_piece_size()).to(DEVICE)
state = torch.load(CKPT_IN, map_location='cpu', weights_only=True)
model.load_state_dict(state)
del state
assert model.pos_emb.num_embeddings == 384
print('Loaded:', CKPT_IN, '| parameters:', sum(p.numel() for p in model.parameters()))


## 학습 설정과 손실 검사
배치별 동적 패딩, 질문/컨텍스트/패딩 마스킹, 다음 토큰 shift를 적용합니다. 검증 손실은 답변 토큰 수로 가중합니다.


In [ ]:
from torch.utils.data import DataLoader
CONFIG = {
    'task': TASK, 'train_window_target': 256, 'model_block_size': 384,
    'batch_size': 32, 'max_epochs': 2, 'lr': 2e-5,
    'patience': 1, 'min_delta': 1e-4, 'warmup_ratio': .05,
    'weight_decay': .01, 'grad_clip': 1., 'seed': SEED,
    'bundle_sha256': EXPECTED_BUNDLE_SHA256,
}
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
loader_generator = torch.Generator().manual_seed(SEED)
collate = partial(collate_samples, pad_id=max(0, sp.pad_id()), max_length=384)
train_dl = DataLoader(train_samples, batch_size=CONFIG['batch_size'], shuffle=True,
    generator=loader_generator, drop_last=False, collate_fn=collate, num_workers=0)
val_dl = DataLoader(val_samples, batch_size=CONFIG['batch_size'], shuffle=False,
    collate_fn=collate, num_workers=0)
assert len(train_dl) > 0 and len(val_dl) > 0
x, y = next(iter(val_dl))
assert x.shape == y.shape and x.shape[1] <= 384
assert y[:,1:].ne(-100).any()
with torch.no_grad():
    model.eval()
    smoke_loss = causal_loss(model, x[:1].to(DEVICE), y[:1].to(DEVICE))
assert torch.isfinite(smoke_loss), 'Invalid masked loss'
print('Finite loss probe:', smoke_loss.item(), '| steps per epoch:', len(train_dl))
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))


## 멀티턴 학습 검사와 고정 평가셋
동일한 평가셋으로 기존 모델과 best 모델을 비교합니다. 이전 대화를 제거한 likelihood, 대화 이력만 바꾼 지시대명사 대조쌍, 단일턴 퇴행 및 뉴스 거절 표현을 기록합니다. 자동 점수는 의미 정확도의 증명이 아닙니다.


In [ ]:
"""Reproducible before/best evaluation; standalone source embedded in Colab.

History-ablation NLL and lexical anchor/F1 scores are diagnostic proxies, not
proof of semantic understanding. Actual model metrics are measured only when
Colab executes this code; local data audits never fabricate model performance.
"""
import csv
from collections import Counter,defaultdict
from datetime import datetime,timezone
import hashlib
import json
import math
from pathlib import Path
import platform
import random
import re
import time

REFERENCE_PATTERN=re.compile(r'그\s*(?:회사|거|건|럼|내용|소식)|거기|방금|아까|앞서|더 자세히')
REFUSAL_PATTERN=re.compile(r'정보가 없습니다|확인할 수 없습니다|알 수 없습니다|뉴스에 없|제공되지')
PAIR_PATTERN=re.compile(r'^질문: ([^\n]*)\n답변: ([^\n]*)(?:\n|$)',re.MULTILINE)


def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()


def protocol_sha256(cases):
    return hashlib.sha256(json.dumps(cases,ensure_ascii=False,sort_keys=True).encode()).hexdigest()


def unpack_turns(text):
    matches=list(PAIR_PATTERN.finditer(text))
    if not matches:raise ValueError('No complete QA turns')
    return text[:matches[0].start()],[(m.group(1),m.group(2)) for m in matches]


def audit_multiturn(records):
    counts=Counter();histogram=Counter()
    for r in records:
        _,pairs=unpack_turns(r['text']);n=len(pairs)
        counts['samples']+=1;histogram[str(n)]+=1
        counts['turns']+=n;counts['later_turns']+=max(0,n-1)
        counts['multiturn_samples']+=n>=2
        counts['reference_later_questions']+=sum(bool(REFERENCE_PATTERN.search(q)) for q,a in pairs[1:])
    return {**counts,'turn_count_histogram':dict(histogram),
            'multiturn_sample_ratio':counts['multiturn_samples']/max(1,counts['samples']),
            'later_reference_expression_ratio':counts['reference_later_questions']/max(1,counts['later_turns']),
            'caveat':'Reference-expression matches are lexical signals, not semantic correctness.'}


def controlled_cases(task):
    cases=[]
    if task=='smalltalk':
        for index,(a,b) in enumerate([('등산','수영'),('독서','요리'),('사진 촬영','자전거')]):
            for side,(target,foil) in enumerate([(a,b),(b,a)]):
                q='방금 내가 했다고 말한 활동은 뭐였지?'
                history=f"질문: 나 오늘 '{target}' 활동을 했어.\n답변: 오늘 {target} 활동을 했구나. 어땠어?\n"
                cases.append({'case_id':f'controlled-{index}-{side}','category':'controlled_reference',
                    'source_key':'controlled_synthetic:'+str(index),'question':q,'reference_answer':target,
                    'prompt':history+f'질문: {q}\n답변: ','no_history_prompt':f'질문: {q}\n답변: ',
                    'context':'','expected_anchor':target,'foil_anchor':foil,'pair_id':str(index),'expected_refusal':None})
    else:
        for index,(a,b) in enumerate([('포장 설비','물류 창고'),('연구 시설','생산 장비'),('배송 차량','보안 시스템')]):
            context=f'제목: 점검 안내(평가용 가상 자료)\n내용: A기업은 {a}를 점검했다. B기업은 {b}를 점검했다.'
            head='참고 뉴스:\n'+context+'\n\n';q='그 회사가 점검한 대상은 뭐야?'
            for side,(company,target,foil) in enumerate([('A기업',a,b),('B기업',b,a)]):
                particle='을' if (ord(target[-1])-0xAC00)%28 else '를'
                history=f'질문: {company}의 소식은?\n답변: {company}은 {target}{particle} 점검했습니다.\n'
                cases.append({'case_id':f'controlled-{index}-{side}','category':'controlled_reference',
                    'source_key':'controlled_synthetic:'+str(index),'question':q,'reference_answer':target,
                    'prompt':head+history+f'질문: {q}\n답변: ','no_history_prompt':head+f'질문: {q}\n답변: ',
                    'context':context,'expected_anchor':target,'foil_anchor':foil,'pair_id':str(index),'expected_refusal':False})
    return cases


def build_evaluation_cases(records,task,limit=16,seed=20261024):
    candidates=defaultdict(list)
    for r in sorted(records,key=lambda r:r['conversation_id']):
        head,pairs=unpack_turns(r['text'])
        for index,(q,a) in enumerate(pairs):
            if index==0 and REFERENCE_PATTERN.search(q):continue
            if index>0 and not REFERENCE_PATTERN.search(q):continue
            category='single_turn' if index==0 else 'multi_turn'
            current=f'질문: {q}\n답변: '
            history=''.join(f'질문: {x}\n답변: {y}\n' for x,y in pairs[:index])
            candidates[category].append({'case_id':r['conversation_id']+':turn'+str(index),
                'category':category,'source_key':r['source_key'],'question':q,'reference_answer':a,
                'prompt':head+history+current,'no_history_prompt':head+current if index else None,
                'context':head,'expected_anchor':None,'foil_anchor':None,'pair_id':None,
                'expected_refusal':bool(REFUSAL_PATTERN.search(a)) if task=='news' else None})
    rng=random.Random(seed);result=[]
    for category in ['single_turn','multi_turn']:
        pool=candidates[category];rng.shuffle(pool);chosen=[];used=set()
        # Include both news refusal and factual follow-ups in the frozen probe set.
        if task=='news' and category=='multi_turn':
            for c in pool:
                if c['expected_refusal'] and c['source_key'] not in used:
                    chosen.append(c);used.add(c['source_key'])
                    if len(chosen)>=max(1,round(limit*.28)):break
        for c in pool:
            if len(chosen)>=limit:break
            if c['source_key'] in used:continue
            if task=='news' and category=='multi_turn' and c['expected_refusal']:continue
            chosen.append(c);used.add(c['source_key'])
        result.extend(chosen)
    return result+controlled_cases(task)


def lexical_f1(answer,reference):
    a=re.sub(r'\s+','',answer);b=re.sub(r'\s+','',reference)
    aa=Counter(a[i:i+2] for i in range(max(1,len(a)-1))) if a else Counter()
    bb=Counter(b[i:i+2] for i in range(max(1,len(b)-1))) if b else Counter()
    common=sum((aa&bb).values())
    return 2*common/(sum(aa.values())+sum(bb.values())) if aa and bb else 0.


def repetition_ratio(answer):
    tokens=answer.split()
    grams=[tuple(tokens[i:i+4]) for i in range(max(0,len(tokens)-3))]
    return (len(grams)-len(set(grams)))/len(grams) if grams else 0.


def last_answer_nll(model,sp,prompt,answer,device,mask_fn):
    import torch
    import torch.nn.functional as F
    ids=sp.encode(prompt+answer+'\n',out_type=int)
    if len(ids)>384:raise ValueError('Evaluation target exceeds model window')
    mask=mask_fn(ids,sp)
    end=max(i for i,keep in enumerate(mask) if keep)
    start=end
    while start>0 and mask[start-1]:start-=1
    labels=[token if start<=i<=end else -100 for i,token in enumerate(ids)]
    x=torch.tensor([ids],device=device);y=torch.tensor([labels],device=device)
    with torch.no_grad():
        logits,_=model(x)
        count=int(y[:,1:].ne(-100).sum())
        loss=F.cross_entropy(logits[:,:-1,:].reshape(-1,logits.shape[-1]),y[:,1:].reshape(-1),ignore_index=-100,reduction='sum')
    return float(loss.item()/count),count


def percentile(values,fraction):
    if not values:return None
    ordered=sorted(values);position=(len(values)-1)*fraction;lo=int(position);hi=math.ceil(position)
    return ordered[lo]+(ordered[hi]-ordered[lo])*(position-lo)


def summarize_rows(rows):
    def weighted(items,key):
        good=[r for r in items if r.get(key) is not None]
        n=sum(r['answer_tokens'] for r in good)
        return sum(r[key]*r['answer_tokens'] for r in good)/n if n else None
    nll=weighted(rows,'answer_nll')
    history=[r for r in rows if r.get('no_history_nll') is not None]
    gain=sum((r['no_history_nll']-r['answer_nll'])*r['answer_tokens'] for r in history)/sum(r['answer_tokens'] for r in history) if history else None
    anchors=[r['anchor_hit'] for r in rows if r.get('anchor_hit') is not None]
    pairs=defaultdict(list)
    for r in rows:
        if r.get('pair_id') is not None:pairs[r['pair_id']].append(r['anchor_hit'])
    complete_pairs=[values for values in pairs.values() if len(values)==2]
    positives=[r for r in rows if r.get('expected_refusal') is True]
    negatives=[r for r in rows if r.get('expected_refusal') is False]
    seconds=sum(r['latency_seconds'] for r in rows)
    result={'cases':len(rows),'answer_nll':nll,'answer_perplexity':math.exp(nll) if nll is not None and nll<700 else None,
        'single_turn_answer_nll':weighted([r for r in rows if r['category']=='single_turn'],'answer_nll'),
        'multi_turn_answer_nll':weighted([r for r in rows if r['category']=='multi_turn'],'answer_nll'),
        'history_gain_nll':gain,
        'history_helped_case_rate':sum(r['answer_nll']<r['no_history_nll'] for r in history)/len(history) if history else None,
        'controlled_anchor_match_rate':sum(anchors)/len(anchors) if anchors else None,
        'controlled_pair_match_rate':sum(all(v) for v in complete_pairs)/len(complete_pairs) if complete_pairs else None,
        'reference_character_bigram_f1':sum(r['reference_f1'] for r in rows)/len(rows) if rows else None,
        'repeated_4gram_ratio':sum(r['repetition_ratio'] for r in rows)/len(rows) if rows else None,
        'empty_answer_rate':sum(r['empty_answer'] for r in rows)/len(rows) if rows else None,
        'refusal_recall_proxy':sum(r['predicted_refusal'] for r in positives)/len(positives) if positives else None,
        'false_refusal_rate_proxy':sum(r['predicted_refusal'] for r in negatives)/len(negatives) if negatives else None,
        'generation_latency_p50_seconds':percentile([r['latency_seconds'] for r in rows],.5),
        'generation_latency_p95_seconds':percentile([r['latency_seconds'] for r in rows],.95),
        'generated_tokens_per_second':sum(r['generated_tokens'] for r in rows)/seconds if seconds else None}
    for category in ['multi_turn','controlled_reference']:
        selected=[r for r in history if r['category']==category]
        result[category+'_history_gain_nll']=sum((r['no_history_nll']-r['answer_nll'])*r['answer_tokens'] for r in selected)/sum(r['answer_tokens'] for r in selected) if selected else None
    return result


def measure_model(model,sp,cases,device,mask_fn,task,validation_nll,max_new_tokens=80,seed=20261024):
    import torch
    gpu=str(device).startswith('cuda')
    def sync():
        if gpu:torch.cuda.synchronize()
    stop={sp.piece_to_id('\n')}
    if sp.eos_id()>=0:stop.add(sp.eos_id())
    rows=[];was_training=model.training;model.eval()
    devices=[torch.cuda.current_device()] if gpu else []
    with torch.random.fork_rng(devices=devices),torch.no_grad():
        # Warm up generation separately so initialization is not counted as latency.
        if cases:
            warm=torch.tensor([sp.encode(cases[0]['prompt'])],device=device)
            model.generate(warm,2,stop_tokens=stop,temperature=1.,top_k=1,repetition_penalty=1.3);sync()
        for index,case in enumerate(cases):
            torch.manual_seed(seed+index)
            nll,count=last_answer_nll(model,sp,case['prompt'],case['reference_answer'],device,mask_fn)
            ablated=None
            if case['no_history_prompt'] is not None:
                ablated,_=last_answer_nll(model,sp,case['no_history_prompt'],case['reference_answer'],device,mask_fn)
            ids=sp.encode(case['prompt'],out_type=int)
            x=torch.tensor([ids],device=device);sync();started=time.perf_counter()
            generated=model.generate(x,max_new_tokens,stop_tokens=stop,temperature=1.,top_k=1,repetition_penalty=1.3)
            sync();elapsed=time.perf_counter()-started
            output=generated[0,len(ids):].tolist();answer=sp.decode(output).strip()
            anchor=case['expected_anchor'];foil=case['foil_anchor']
            compact=lambda value:re.sub(r'\s+','',value)
            hit=(compact(anchor) in compact(answer) and compact(foil) not in compact(answer)) if anchor is not None else None
            rows.append({**case,'generated_answer':answer,'answer_nll':nll,'answer_tokens':count,
                'no_history_nll':ablated,'generated_tokens':len(output),'latency_seconds':elapsed,
                'reference_f1':lexical_f1(answer,case['reference_answer']),'repetition_ratio':repetition_ratio(answer),
                'empty_answer':not bool(answer),'predicted_refusal':bool(REFUSAL_PATTERN.search(answer)),'anchor_hit':hit})
    model.train(was_training)
    aggregate=summarize_rows(rows)
    aggregate.update(validation_answer_nll=validation_nll,validation_answer_perplexity=math.exp(validation_nll) if validation_nll<700 else None)
    return {'task':task,'measured_at':datetime.now(timezone.utc).isoformat(),'protocol_sha256':protocol_sha256(cases),
            'generation':{'top_k':1,'temperature':1.,'repetition_penalty':1.3,'max_new_tokens':max_new_tokens,'seed':seed},
            'environment':{'torch':str(torch.__version__),'cuda':torch.version.cuda,'device':str(device),
                'gpu':torch.cuda.get_device_name() if gpu else None,'platform':platform.platform()},
            'aggregate':aggregate,'rows':rows,
            'limitations':['Character F1, refusal strings and anchor matches are lexical proxies, not semantic accuracy.',
                'History-ablation gain measures likelihood sensitivity, not proof of correct reference resolution.',
                'Controlled cases are synthetic evaluation fixtures only; they are never training data.',
                'Generation latency is a warm batch-1 probe, not an end-to-end serving benchmark.']}


def save_json(path,value):
    Path(path).write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')


def save_comparison(folder,before,after):
    folder=Path(folder)
    if before['protocol_sha256']!=after['protocol_sha256']:raise ValueError('Evaluation protocols differ')
    if [r['case_id'] for r in before['rows']]!=[r['case_id'] for r in after['rows']]:raise ValueError('Evaluation case order differs')
    if before.get('generation')!=after.get('generation'):raise ValueError('Generation configurations differ')
    old_env,new_env=before.get('environment',{}),after.get('environment',{})
    timing_comparable=bool(old_env and new_env) and all(old_env.get(k)==new_env.get(k) for k in ['torch','cuda','device','gpu'])
    keys=sorted(set(before['aggregate'])|set(after['aggregate']))
    delta={key:after['aggregate'][key]-before['aggregate'][key] if isinstance(before['aggregate'].get(key),(int,float)) and isinstance(after['aggregate'].get(key),(int,float)) else None for key in keys}
    result={'protocol_sha256':before['protocol_sha256'],'before':before['aggregate'],'after':after['aggregate'],
            'delta_after_minus_before':delta,'semantic_accuracy':None,'manual_review_required':True,'timing_environment_matches':timing_comparable,
            'note':'Positive history_gain_nll means history lowers target NLL; all lexical scores require semantic review.'}
    save_json(folder/'comparison.json',result)
    with (folder/'comparison.csv').open('w',encoding='utf-8-sig',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=['metric','before','after','delta_after_minus_before']);writer.writeheader()
        for key in keys:writer.writerow({'metric':key,'before':before['aggregate'].get(key),'after':after['aggregate'].get(key),'delta_after_minus_before':delta[key]})
    fields=['case_id','category','source_key','prompt','reference_answer','before_answer','after_answer','before_semantic_pass','after_semantic_pass']
    with (folder/'manual_review.csv').open('w',encoding='utf-8-sig',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader()
        for old,new in zip(before['rows'],after['rows']):
            writer.writerow({**{key:old[key] for key in fields[:5]},'before_answer':old['generated_answer'],
                             'after_answer':new['generated_answer'],'before_semantic_pass':'','after_semantic_pass':''})
    return result


In [ ]:
EVALUATION_CASES = build_evaluation_cases(val_records, TASK, limit=16, seed=SEED)
EVALUATION_SHA256 = protocol_sha256(EVALUATION_CASES)
CONFIG['evaluation_protocol_sha256'] = EVALUATION_SHA256
CONFIG['evaluation_code_sha256'] = '118cad5f21100e0b5bc9affad0392ab67241a3d5ad1c8fa16458cd2226c97805'
CONFIG['evaluation_max_new_tokens'] = 96 if TASK == 'news' else 80
DATASET_AUDIT = {'train': audit_multiturn(train_records), 'val': audit_multiturn(val_records)}
assert DATASET_AUDIT['train']['multiturn_samples'] > 0, 'No multi-turn training examples'
assert DATASET_AUDIT['val']['multiturn_samples'] > 0, 'No multi-turn validation examples'
print('Multi-turn training audit:', json.dumps(DATASET_AUDIT, ensure_ascii=False, indent=2))
print('Frozen evaluation cases:', len(EVALUATION_CASES), '| protocol:', EVALUATION_SHA256)


## GPU 학습 및 Drive 저장
A100 또는 사용 가능한 Colab GPU로 실행하세요. 매 에폭 best 가중치와 재개 상태를 Drive에 저장합니다. 중단 후 `RESUME_PATH`에 해당 실행의 `last.resume.pt`를 지정하면 에폭 경계부터 재개합니다.


In [ ]:
assert DEVICE == 'cuda', 'Use a Colab GPU runtime; CPU training is intentionally disabled'
# Set this only to this notebook's own trusted resume file, never an untrusted pickle.
RESUME_PATH = None
if RESUME_PATH:
    RUN_DIR = Path(RESUME_PATH).parent
else:
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    RUN_DIR = MODELS_DIR / (TASK + '_' + run_id)
    RUN_DIR.mkdir(parents=True, exist_ok=False)
CKPT_OUT = RUN_DIR / OUTPUT_NAME
LAST_STATE = RUN_DIR / 'last.resume.pt'
HISTORY_PATH = RUN_DIR / 'training_history.json'
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
total_steps = len(train_dl) * CONFIG['max_epochs']
warmup = max(1, int(total_steps * CONFIG['warmup_ratio']))
def lr_factor(step):
    if step < warmup: return (step + 1) / warmup
    progress = (step - warmup) / max(1, total_steps - warmup)
    return .5 * (1 + math.cos(math.pi * min(1., progress)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
start_epoch = 0; stale = 0; history = []
baseline_loss = None
best = None
if RESUME_PATH:
    resume = torch.load(RESUME_PATH, map_location='cpu', weights_only=False)
    assert resume['config'] == CONFIG, 'Resume config/data mismatch'
    assert CKPT_OUT.is_file(), 'Best model is missing from the resumed run'
    model.load_state_dict(resume['model'])
    optimizer.load_state_dict(resume['optimizer']); scheduler.load_state_dict(resume['scheduler'])
    start_epoch = resume['next_epoch']; best = resume['best']; stale = resume['stale']
    baseline_loss = resume['baseline_loss']; history = resume['history']
    random.setstate(resume['python_rng']); torch.set_rng_state(resume['torch_rng'])
    torch.cuda.set_rng_state_all(resume['cuda_rng'])
    loader_generator.set_state(resume['loader_rng'])
else:
    # Always measure the actual original checkpoint, including cell re-runs.
    model.load_state_dict(torch.load(CKPT_IN, map_location='cpu', weights_only=True))
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    loader_generator.manual_seed(SEED)
    baseline_loss = evaluate_token_weighted(model, val_dl, DEVICE)
    best = baseline_loss
    # Preserve the baseline as best if no fine-tuned epoch improves validation.
    save_torch_state({k: v.detach().cpu() for k,v in model.state_dict().items()}, CKPT_OUT)
    (RUN_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')

BEFORE_PATH = RUN_DIR / 'before_metrics.json'
AFTER_PATH = RUN_DIR / 'after_metrics.json'
if RESUME_PATH:
    assert BEFORE_PATH.is_file(), 'This run has no original before measurement; use a fresh run for a valid comparison'
    before_metrics = json.loads(BEFORE_PATH.read_text())
    assert before_metrics['protocol_sha256'] == EVALUATION_SHA256
else:
    save_json(RUN_DIR / 'evaluation_cases.json', EVALUATION_CASES)
    save_json(RUN_DIR / 'dataset_audit.json', DATASET_AUDIT)
    before_metrics = measure_model(model, sp, EVALUATION_CASES, DEVICE, answer_mask, TASK,
        baseline_loss, max_new_tokens=CONFIG['evaluation_max_new_tokens'], seed=SEED)
    before_metrics['checkpoint_sha256'] = file_sha256(CKPT_IN)
    before_metrics['evaluation_code_sha256'] = CONFIG['evaluation_code_sha256']
    save_json(BEFORE_PATH, before_metrics)
print('Baseline val loss:', baseline_loss, '| output:', CKPT_OUT)
print('Persisted original-model metrics:', BEFORE_PATH)
for epoch in range(start_epoch, CONFIG['max_epochs']):
    if stale >= CONFIG['patience']: break
    model.train(); started = time.time(); running = 0.; targets = 0
    optimizer.zero_grad(set_to_none=True)
    for step, (x,y) in enumerate(train_dl):
        x=x.to(DEVICE); y=y.to(DEVICE)
        loss=causal_loss(model,x,y)
        if not torch.isfinite(loss): raise ValueError('Non-finite loss; stop without replacing best')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG['grad_clip'])
        optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
        count=int(y[:,1:].ne(-100).sum()); running+=loss.item()*count; targets+=count
        if step % 100 == 0: print('epoch',epoch+1,'step',step,'loss',round(loss.item(),4),flush=True)
    val_loss=evaluate_token_weighted(model,val_dl,DEVICE)
    improved=val_loss < best-CONFIG['min_delta']
    if improved:
        best=val_loss; stale=0
        save_torch_state({k:v.detach().cpu() for k,v in model.state_dict().items()},CKPT_OUT)
    else: stale+=1
    history.append({'epoch':epoch+1,'train_loss':running/targets,'val_loss':val_loss,
                    'improved':improved,'seconds':time.time()-started})
    HISTORY_PATH.write_text(json.dumps(history,indent=2),encoding='utf-8')
    save_torch_state({'model':model.state_dict(),'optimizer':optimizer.state_dict(),
        'scheduler':scheduler.state_dict(),'next_epoch':epoch+1,'best':best,'stale':stale,
        'baseline_loss':baseline_loss,'history':history,'config':CONFIG,
        'python_rng':random.getstate(),'torch_rng':torch.get_rng_state(),
        'cuda_rng':torch.cuda.get_rng_state_all(),'loader_rng':loader_generator.get_state()},LAST_STATE)
    print(history[-1], '| best:',CKPT_OUT,flush=True)

model.load_state_dict(torch.load(CKPT_OUT,map_location='cpu',weights_only=True))
print('Reloaded best state:',CKPT_OUT,'| baseline:',baseline_loss,'best:',best)
best_sha256 = file_sha256(CKPT_OUT)
if AFTER_PATH.is_file():
    after_metrics = json.loads(AFTER_PATH.read_text())
    assert after_metrics['checkpoint_sha256'] == best_sha256, 'Stored evaluation does not match best model'
else:
    after_loss = evaluate_token_weighted(model, val_dl, DEVICE)
    after_metrics = measure_model(model, sp, EVALUATION_CASES, DEVICE, answer_mask, TASK,
        after_loss, max_new_tokens=CONFIG['evaluation_max_new_tokens'], seed=SEED)
    after_metrics['checkpoint_sha256'] = best_sha256
    after_metrics['evaluation_code_sha256'] = CONFIG['evaluation_code_sha256']
    save_json(AFTER_PATH, after_metrics)
comparison = save_comparison(RUN_DIR, before_metrics, after_metrics)
save_json(RUN_DIR / 'training_summary.json', {
    'baseline_validation_nll': baseline_loss, 'best_validation_nll': best,
    'delta_best_minus_baseline': best-baseline_loss, 'epochs_completed': len(history),
    'optimizer_steps_completed': len(history)*len(train_dl),
    'training_seconds': sum(h['seconds'] for h in history),
    'config': CONFIG, 'best_checkpoint_sha256': best_sha256,
    'comparison_file': str(RUN_DIR / 'comparison.json')})
print('Saved before/after metrics and comparison:', RUN_DIR / 'comparison.csv')
if best >= baseline_loss-CONFIG['min_delta']:
    print('Validation did not improve: best output retains the initial weights; do not claim an improvement.')


## best 체크포인트 비교
마지막 에폭이 아닌 검증상 best 가중치를 사용합니다. 근거 없는 수치를 만들거나 단일턴이 나빠지면 채택하지 마세요. 전후 수치와 원문 답변은 Drive의 comparison.json/CSV와 manual_review.csv에 저장합니다. 자동 점수는 일반화 성능을 보장하지 않습니다.


In [ ]:
print('Before / best / after-minus-before:')
for key, value in comparison['delta_after_minus_before'].items():
    print(key, '|', comparison['before'].get(key), '|', comparison['after'].get(key), '| delta:', value)
print('Semantic accuracy is not automatically certified. Review manual_review.csv.')
print('Full artifacts:', RUN_DIR)
for old,new in zip(before_metrics['rows'], after_metrics['rows']):
    if old['category']=='controlled_reference':
        print(old['case_id'], '| expected anchor:', old['expected_anchor'],
              '| before:', old['generated_answer'], '| best:', new['generated_answer'])
